# E-Commerce Behavior: Marketing Funnel and Conversion Analysis

## Project Overview
This notebook provides a comprehensive analysis of customer behavior on a large multi-category e-commerce platform. We analyze the full dataset of over 67 million records to understand the customer journey from initial view to final purchase.

### Key Objectives:
1. **Funnel Analysis**: Track user drop-off between View, Cart, and Purchase stages.
2. **Temporal Trends**: Identify peak activity hours and conversion patterns throughout the week.
3. **Price Impact**: Analyze how product pricing influences purchase probability.
4. **Brand & Category Performance**: Identify top-performing brands and product categories.

---

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
import time

# Configuration
INPUT_PATH = "../data/cleaned_data.csv"
CHUNK_SIZE = 2500000
pd.options.plotting.backend = "plotly"

## 1. Data Processing Engine
Given the scale of the dataset (approx. 7.5GB cleaned), we employ a chunked processing strategy to maintain memory efficiency on a 16GB RAM system. We optimize for speed by using string-slicing for temporal features and vectorized aggregations.

In [2]:
# Accumulators
funnel_users = {'view': set(), 'cart': set(), 'purchase': set()}
event_counts = {'view': 0, 'cart': 0, 'purchase': 0}
hourly_users = {i: set() for i in range(24)}
daily_users = {day: set() for day in ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']}

price_brackets = ['0-25', '26-50', '51-100', '101-250', '251-500', '501-1000', '1000+']
price_views = {bracket: 0 for bracket in price_brackets}
price_purchases = {bracket: 0 for bracket in price_brackets}

# Brand and Category tracking
brand_events = {}
brand_purchases = {}
category_views = {}

total_users_set = set()
total_rows = 0
start_time = time.time()
date_to_day_cache = {}

print(f"Processing dataset chunks...")

use_cols = ['event_time', 'event_type', 'user_id', 'price', 'brand', 'category_code']

for chunk in pd.read_csv(INPUT_PATH, chunksize=CHUNK_SIZE, usecols=use_cols):
    # Temporal Optimization
    chunk['h'] = chunk['event_time'].str[11:13].astype(np.int8)
    dates = chunk['event_time'].str[0:10]
    for d in dates.unique():
        if d not in date_to_day_cache: date_to_day_cache[d] = pd.to_datetime(d).day_name()
    chunk['d'] = dates.map(date_to_day_cache)
    
    # Funnel Updates
    total_users_set.update(chunk['user_id'].unique())
    for etype in ['view', 'cart', 'purchase']:
        mask = chunk['event_type'] == etype
        funnel_users[etype].update(chunk.loc[mask, 'user_id'].unique())
        event_counts[etype] += mask.sum()
    
    # Temporal Groups
    for h, ids in chunk.groupby('h')['user_id'].unique().items(): hourly_users[h].update(ids)
    for d, ids in chunk.groupby('d')['user_id'].unique().items(): daily_users[d].update(ids)
    
    # Price Analysis
    chunk['pr'] = pd.cut(chunk['price'], bins=[0, 25, 50, 100, 250, 500, 1000, float('inf')], labels=price_brackets)
    v_vc = chunk.loc[chunk['event_type'] == 'view', 'pr'].value_counts()
    p_vc = chunk.loc[chunk['event_type'] == 'purchase', 'pr'].value_counts()
    for br in price_brackets:
        if br in v_vc: price_views[br] += v_vc[br]
        if br in p_vc: price_purchases[br] += p_vc[br]
        
    # Brand & Category Tracking
    be = chunk['brand'].value_counts()
    for b, count in be.items(): brand_events[b] = brand_events.get(b, 0) + count
    
    bp = chunk[chunk['event_type'] == 'purchase']['brand'].value_counts()
    for b, count in bp.items(): brand_purchases[b] = brand_purchases.get(b, 0) + count
    
    cv = chunk[chunk['event_type'] == 'view']['category_code'].value_counts()
    for c, count in cv.items(): category_views[c] = category_views.get(c, 0) + count

    total_rows += len(chunk)
    # Professional print without emojis
    print(f"  Progress: {total_rows/1_000_000:.1f}M rows processed...")

print(f"\nAnalysis complete! Total rows: {total_rows:,} | Total time: {(time.time() - start_time)/60:.2f} minutes.")

Processing dataset chunks...
  Progress: 2.5M rows processed...
  Progress: 5.0M rows processed...
  Progress: 7.5M rows processed...
  Progress: 10.0M rows processed...
  Progress: 12.5M rows processed...
  Progress: 15.0M rows processed...
  Progress: 17.5M rows processed...
  Progress: 20.0M rows processed...
  Progress: 22.5M rows processed...
  Progress: 25.0M rows processed...
  Progress: 27.5M rows processed...
  Progress: 30.0M rows processed...
  Progress: 32.5M rows processed...
  Progress: 35.0M rows processed...
  Progress: 37.5M rows processed...
  Progress: 40.0M rows processed...
  Progress: 42.5M rows processed...
  Progress: 45.0M rows processed...
  Progress: 47.5M rows processed...
  Progress: 50.0M rows processed...
  Progress: 52.5M rows processed...
  Progress: 55.0M rows processed...
  Progress: 57.5M rows processed...
  Progress: 60.0M rows processed...
  Progress: 62.5M rows processed...
  Progress: 65.0M rows processed...
  Progress: 67.4M rows processed...

A

## 2. Marketing Funnel Visualization
The purchase funnel reveals how many unique users transition through the customer journey.

In [3]:
fig_funnel = go.Figure(go.Funnel(
    y = ["Product Views", "Add to Carts", "Confirmed Purchases"],
    x = [len(funnel_users[s]) for s in ['view', 'cart', 'purchase']],
    textinfo = "value+percent initial",
    marker = {"color": ["royalblue", "darkorange", "forestgreen"]}))

fig_funnel.update_layout(title_text="User Conversion Funnel", template="plotly_white")
fig_funnel.show()

## 3. Temporal Engagement Trends
Understanding when users are most active allows for better marketing campaign timing.

In [4]:
hour_labels = [f"{i:02d}:00" for i in range(24)]
hour_vals = [len(hourly_users[i]) for i in range(24)]
fig_h = px.line(x=hour_labels, y=hour_vals, title="Hourly User Activity", labels={'x':'Time of Day','y':'Unique Users'})
fig_h.show()

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_counts = [len(daily_users[d]) for d in day_order]
fig_d = px.bar(x=day_order, y=day_counts, title="Daily Activity Distribution", labels={'x':'Day','y':'Users'}, color=day_counts)
fig_d.show()

## 4. Price Elasticity and Conversion Rates
How does the price of a product impact its conversion rate?

In [5]:
cr_vals = [(price_purchases[b]/price_views[b]*100) if price_views[b]>0 else 0 for b in price_brackets]
fig_p = px.bar(x=price_brackets, y=cr_vals, title="Conversion Rate by Price Bracket (%)", labels={'x':'Price Bracket ($)','y':'Conversion Rate (%)'}, color=cr_vals)
fig_p.show()

## 5. Brand and Category Performance
We identify the market leaders in terms of volume and conversion.

In [6]:
# Top 10 Brands
top_brands = pd.Series(brand_events).sort_values(ascending=False).head(10)
fig_b = px.bar(x=top_brands.index, y=top_brands.values, title="Top 10 Brands by Event Volume", labels={'x':'Brand','y':'Total Events'})
fig_b.show()

# Top 10 Categories
top_cats = pd.Series(category_views).sort_values(ascending=False).head(10)
fig_c = px.pie(names=top_cats.index, values=top_cats.values, title="Top 10 Product Categories (by View Volume)")
fig_c.show()

# Brand Conversion for Top Selling Brands
top_purch_brands = pd.Series(brand_purchases).sort_values(ascending=False).head(10).index
brand_cr = {b: (brand_purchases.get(b,0) / brand_events.get(b,1) * 100) for b in top_purch_brands}
fig_bcr = px.bar(x=list(brand_cr.keys()), y=list(brand_cr.values()), title="Conversion Rate for Top Selling Brands (%)", labels={'x':'Brand','y':'CR (%)'})
fig_bcr.show()

## 6. Executive Summary

Based on the analysis of **67.4 million** records, here are the key findings:

- **Funnel Performance**: While view counts are massive, only a fraction of users convert. The View-to-Cart drop-off is the most critical area for optimization.
- **Peak Hours**: Activity peaks between **13:00 and 16:00**, suggesting prime window for push notifications or live offers.
- **Price Sweet Spot**: Products in the **101-250$** range show consistently high conversion markers, possibly indicating highly targeted electronics or home appliance audiences.
- **Brand Dominance**: A few key brands dominate both views and purchases, showing high brand loyalty or market ubiquity.